In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00


In [3]:
import json

dataset = [
    {"instruction": "What is the capital of India?", "input": "", "output": "The capital of India is New Delhi. It serves as the seat of all three branches of the Government of India."},
    {"instruction": "Name the seven union territories of India.", "input": "", "output": "The seven union territories of India are: Andaman and Nicobar Islands, Chandigarh, Dadra and Nagar Haveli and Daman and Diu, Delhi (NCT), Jammu and Kashmir, Ladakh, and Lakshadweep."},
    {"instruction": "What is the significance of the Ganges river in Indian culture?", "input": "", "output": "The Ganges is the most sacred river in Hinduism and is personified as the goddess Ganga. It is believed to cleanse sins and aid in attaining moksha. Millions of pilgrims bathe in its waters, especially at cities like Varanasi, Haridwar, and Allahabad during festivals like Kumbh Mela."},
    {"instruction": "What are the major classical dance forms of India?", "input": "", "output": "India has eight major classical dance forms recognised by the Sangeet Natak Akademi: Bharatanatyam (Tamil Nadu), Kathak (North India), Kathakali (Kerala), Kuchipudi (Andhra Pradesh), Manipuri (Manipur), Mohiniyattam (Kerala), Odissi (Odisha), and Sattriya (Assam)."},
    {"instruction": "What is the significance of Diwali?", "input": "", "output": "Diwali, the Festival of Lights, is one of India's most celebrated festivals. It commemorates Lord Rama's return to Ayodhya after 14 years of exile and his victory over Ravana. It is celebrated with lighting of diyas, fireworks, exchange of sweets, and worship of Goddess Lakshmi for prosperity."},
    {"instruction": "Name the major mountain ranges of India.", "input": "", "output": "India's major mountain ranges include the Himalayas in the north (including the Karakoram and Hindu Kush), the Aravalli Range in Rajasthan, the Vindhya Range in central India, the Satpura Range, the Western Ghats along the western coast, and the Eastern Ghats along the eastern coast."},
    {"instruction": "What is the significance of Varanasi?", "input": "", "output": "Varanasi, also known as Kashi or Benaras, is one of the oldest continuously inhabited cities in the world and the holiest city in Hinduism. Located on the banks of the Ganges in Uttar Pradesh, it is a major pilgrimage destination where Hindus believe dying brings moksha. It is also famous for its ghats, silk weaving, and classical music tradition."},
    {"instruction": "What is the national animal of India?", "input": "", "output": "The Bengal Tiger (Panthera tigris tigris) is the national animal of India. It was designated as the national animal in 1973 when Project Tiger was launched to protect the tiger population from extinction. India currently has the largest tiger population in the world."},
    {"instruction": "Describe the geography of Kerala.", "input": "", "output": "Kerala is a state on the southwestern tip of India, known as God's Own Country. It features a narrow coastal strip between the Western Ghats and the Arabian Sea. The state is famous for its backwaters — a network of lagoons, lakes, and canals — lush hill stations like Munnar, tropical rainforests, and over 600km of Arabian Sea coastline."},
    {"instruction": "What is the significance of the Taj Mahal?", "input": "", "output": "The Taj Mahal is a white marble mausoleum in Agra, Uttar Pradesh, built by Mughal Emperor Shah Jahan between 1631 and 1653 in memory of his wife Mumtaz Mahal. It is considered the finest example of Mughal architecture and is a UNESCO World Heritage Site. It attracts millions of tourists annually and is one of the Seven Wonders of the Modern World."},
    {"instruction": "What are the major rivers of South India?", "input": "", "output": "The major rivers of South India include the Godavari, Krishna, Kaveri (Cauvery), Tungabhadra, Periyar, and Mahanadi. Unlike North Indian rivers that originate from the Himalayas, South Indian rivers originate from the Western Ghats and flow eastward into the Bay of Bengal, except the Periyar which flows westward into the Arabian Sea."},
    {"instruction": "What is Carnatic music?", "input": "", "output": "Carnatic music is one of the two main subgenres of Indian classical music, predominantly practiced in the southern states of Tamil Nadu, Karnataka, Andhra Pradesh, Telangana, and Kerala. It is primarily vocal music, emphasising improvisation within a fixed raga and tala framework. Famous composers include Tyagaraja, Muthuswami Dikshitar, and Syama Sastri, known as the Trinity of Carnatic music."},
    {"instruction": "Describe the Thar Desert.", "input": "", "output": "The Thar Desert, also called the Great Indian Desert, is a large arid region in northwestern India and eastern Pakistan. It covers about 200,000 square kilometres, primarily in Rajasthan. Despite being a desert, it is one of the most densely populated deserts in the world. It features sand dunes, rocky terrain, and sparse vegetation. The city of Jaisalmer is a major cultural centre within the Thar Desert."},
    {"instruction": "What is the significance of the Kumbh Mela?", "input": "", "output": "Kumbh Mela is the world's largest religious gathering, held at four sacred river locations in India: Prayagraj, Haridwar, Ujjain, and Nashik. It occurs on a 12-year cycle at each location based on astrological positions. Pilgrims bathe in the sacred rivers believing it cleanses sins and grants moksha. The Prayagraj Kumbh Mela of 2019 attracted an estimated 200 million visitors, making it the largest human gathering in history."},
    {"instruction": "What are the major languages spoken in India?", "input": "", "output": "India has 22 officially recognised languages in its Eighth Schedule, with Hindi and English as official languages of the Union. The most widely spoken languages include Hindi, Bengali, Telugu, Marathi, Tamil, Urdu, Gujarati, Kannada, Odia, and Malayalam. India has hundreds of dialects and is one of the most linguistically diverse countries in the world."},
    {"instruction": "What is the significance of the Sundarbans?", "input": "", "output": "The Sundarbans is the world's largest mangrove forest, spanning the delta of the Ganges, Brahmaputra, and Meghna rivers across India and Bangladesh. Located primarily in West Bengal, it is a UNESCO World Heritage Site and home to the Royal Bengal Tiger. The name comes from the Sundari tree dominant in the region. It provides crucial coastal protection and is a biodiversity hotspot."},
    {"instruction": "Describe the festival of Holi.", "input": "", "output": "Holi is the Hindu festival of colours celebrated at the arrival of spring, typically in March. It commemorates the legend of Prahlad and Holika — symbolising the victory of good over evil. The festival is celebrated by throwing coloured powder and water, singing, dancing, and sharing sweets. Mathura and Vrindavan in Uttar Pradesh have the most famous Holi celebrations, drawing visitors from around the world."},
    {"instruction": "What is the significance of Hampi?", "input": "", "output": "Hampi is a UNESCO World Heritage Site in Karnataka, India, that was the capital of the Vijayanagara Empire from 1336 to 1565 CE — one of the greatest Hindu kingdoms in history. The site contains over 1,600 surviving monuments including temples, palaces, stables, and market streets spread across a dramatic boulder-strewn landscape along the Tungabhadra River."},
    {"instruction": "What is yoga and where did it originate?", "input": "", "output": "Yoga is an ancient Indian practice encompassing physical postures, breathing techniques, meditation, and philosophical principles. It originated in the Indus Valley Civilisation over 5,000 years ago and is described in ancient texts including the Vedas, Upanishads, and Patanjali's Yoga Sutras. The International Day of Yoga on June 21 was established by the UN in 2015 following India's proposal. Rishikesh in Uttarakhand is considered the global capital of yoga."},
    {"instruction": "Name the states that border China.", "input": "", "output": "The Indian states that share a border with China (Tibet Autonomous Region) are: Jammu and Kashmir, Ladakh (union territory), Himachal Pradesh, Uttarakhand, Sikkim, and Arunachal Pradesh. The Line of Actual Control (LAC) forms the de facto boundary between India and China, spanning approximately 3,488 kilometres."},
]

print(f"Dataset created with {len(dataset)} examples")

Dataset created with 20 examples


In [5]:
dataset += [
    {"instruction": "What is the significance of the Western Ghats?", "input": "", "output": "The Western Ghats, also called Sahyadri, is a mountain range running parallel to India's western coast for about 1,600 km through Gujarat, Maharashtra, Goa, Karnataka, Kerala, and Tamil Nadu. It is a UNESCO World Heritage Site and one of the world's eight biodiversity hotspots. It influences the monsoon patterns of the Indian subcontinent and is the source of many major peninsular rivers including the Godavari, Krishna, and Kaveri."},
    {"instruction": "What is the significance of Ajanta and Ellora caves?", "input": "", "output": "Ajanta and Ellora are UNESCO World Heritage Sites in Maharashtra. Ajanta caves are 30 rock-cut Buddhist caves dating from the 2nd century BCE to 480 CE, famous for their magnificent paintings and sculptures depicting the life of Buddha. Ellora caves are 34 monasteries and temples of Buddhist, Hindu, and Jain faiths carved between 600-1000 CE, representing the spirit of religious harmony in ancient India. The Kailasa temple at Ellora is the world's largest monolithic rock excavation."},
    {"instruction": "Describe the geography of the Northeast India.", "input": "", "output": "Northeast India comprises eight states known as the Seven Sisters plus Sikkim: Arunachal Pradesh, Assam, Manipur, Meghalaya, Mizoram, Nagaland, Tripura, and Sikkim. The region is characterised by the Brahmaputra river valley, dense rainforests, and hills. It shares borders with China, Bhutan, Bangladesh, and Myanmar. The region receives some of the highest rainfall in the world — Mawsynram in Meghalaya is often cited as the wettest place on Earth."},
    {"instruction": "What is Bharatanatyam?", "input": "", "output": "Bharatanatyam is one of India's oldest and most widely practised classical dance forms, originating in Tamil Nadu. It evolved from the Devadasi tradition of temple dance and was revived and systematised in the 20th century. The dance combines nritta (pure dance), nritya (expressive dance), and natya (drama). It is characterised by fixed upper torso, bent knees, and intricate footwork, set to Carnatic music. Rukmini Devi Arundale played a crucial role in its revival and global popularisation."},
    {"instruction": "What is the significance of the Brahmaputra river?", "input": "", "output": "The Brahmaputra is one of Asia's major rivers, originating in Tibet as the Yarlung Tsangpo, entering India through Arunachal Pradesh, flowing through Assam, and joining the Ganges in Bangladesh. It is one of the few rivers in the world with a male name. The river is crucial for Assam's ecology and economy, supporting agriculture and fisheries. It is known for its seasonal floods, which while destructive, deposit fertile silt across its floodplains."},
    {"instruction": "What are the major festivals of Tamil Nadu?", "input": "", "output": "Tamil Nadu's major festivals include Pongal — the harvest festival celebrated in January over four days, considered the most important Tamil festival. Other festivals include Karthigai Deepam (festival of lights), Thai Pusam, Navarathri with Golu doll displays, and Chithirai festival in Madurai celebrating the celestial wedding of Meenakshi and Sundareshwara. Tamil Nadu also celebrates national festivals and has a strong tradition of temple festivals at its ancient temples."},
    {"instruction": "What is the significance of Ladakh?", "input": "", "output": "Ladakh is a union territory in northern India, bordered by China and Pakistan. Often called the Land of High Passes, it is one of the highest inhabited plateaus in the world. It features dramatic landscapes including the Himalayas and Karakoram ranges, high-altitude deserts, and glacial lakes like Pangong Tso. Ladakh has a predominantly Buddhist population with Tibetan cultural influences. It was bifurcated from Jammu and Kashmir and made a union territory in 2019."},
    {"instruction": "Describe the Deccan Plateau.", "input": "", "output": "The Deccan Plateau is a large elevated plain covering most of peninsular India, spanning parts of Maharashtra, Karnataka, Andhra Pradesh, and Telangana. It is bounded by the Western Ghats, Eastern Ghats, and Satpura Range. The plateau has an average elevation of 600 metres and is drained by major rivers flowing eastward. It is composed largely of Deccan Traps — volcanic basalt rock — and has historically been the heartland of powerful empires including the Mauryas, Satavahanas, Chalukyas, Rashtrakutas, and Vijayanagara."},
    {"instruction": "What is the significance of Rishikesh?", "input": "", "output": "Rishikesh is a city in Uttarakhand on the banks of the Ganges at the foothills of the Himalayas. It is considered the Yoga Capital of the World and attracts spiritual seekers from across the globe. It is a gateway to the Char Dham pilgrimage circuit. The Beatles visited Rishikesh in 1968 to study Transcendental Meditation, bringing global attention to the city. It is also known for its Ram Jhula and Laxman Jhula suspension bridges and adventure activities like river rafting."},
    {"instruction": "What are the major textile traditions of India?", "input": "", "output": "India has extraordinarily rich textile traditions. Major ones include Banarasi silk from Varanasi, Kanchipuram silk from Tamil Nadu, Chanderi and Maheshwari fabrics from Madhya Pradesh, Pochampally Ikat from Telangana, Bandhani tie-dye from Rajasthan and Gujarat, Pashmina shawls from Kashmir, Jamdani muslin from Bengal, Phulkari embroidery from Punjab, and Kalamkari hand-painted fabric from Andhra Pradesh. Many of these have GI (Geographical Indication) tags protecting their authenticity."},
    {"instruction": "What is the significance of Konark Sun Temple?", "input": "", "output": "The Konark Sun Temple is a 13th century CE temple at Konark in Odisha, dedicated to the Hindu sun god Surya. Built by King Narasimhadeva I of the Eastern Ganga dynasty, it is designed as a colossal chariot with 24 intricately carved wheels pulled by seven horses. The temple is a UNESCO World Heritage Site and one of the finest examples of Kalinga architecture. It is featured on the Indian 10 rupee note and the Odisha state emblem."},
    {"instruction": "Describe the culture of Rajasthan.", "input": "", "output": "Rajasthan, meaning Land of Kings, has one of India's richest cultural heritages. It is known for its magnificent forts and palaces including Amber Fort, Mehrangarh Fort, and City Palace. The state has vibrant folk music traditions including Manganiyar and Langa musicians, folk dances like Ghoomar and Kalbelia, and colourful festivals like Pushkar Camel Fair and Teej. Rajasthani cuisine features dal baati churma, gatte ki sabzi, and laal maas. The state is also famous for its block-printed and tie-dyed textiles and blue pottery of Jaipur."},
    {"instruction": "What is the significance of the Andaman and Nicobar Islands?", "input": "", "output": "The Andaman and Nicobar Islands are a union territory of India comprising 572 islands in the Bay of Bengal. They are strategically located near the Strait of Malacca — a major shipping lane. The islands have a tragic history as the site of the Cellular Jail (Kala Pani) where Indian freedom fighters were imprisoned by the British. They are home to protected indigenous tribes including the Sentinelese — one of the last uncontacted peoples on Earth. The islands have rich biodiversity including coral reefs and tropical rainforests."},
    {"instruction": "What are the major classical music traditions of India?", "input": "", "output": "Indian classical music has two major traditions: Hindustani music of North India and Carnatic music of South India. Hindustani music was influenced by Persian and Mughal elements and features ragas performed in various gharanas (schools). Major instruments include sitar, tabla, sarod, and bansuri. Carnatic music is predominantly vocal with instruments like veena, mridangam, and violin. Both traditions are based on the raga-tala system but differ in performance style, improvisation techniques, and repertoire."},
    {"instruction": "What is the significance of Tirupati?", "input": "", "output": "Tirupati is a city in Andhra Pradesh housing the Tirumala Venkateswara Temple, one of the most visited religious sites in the world. The temple is dedicated to Lord Venkateswara, a form of Vishnu. It receives an average of 50,000-100,000 pilgrims daily and is considered the richest temple in the world by donation income. The famous Tirupati laddoo, offered as prasad, has a GI tag. The temple is managed by the Tirumala Tirupati Devasthanams (TTD)."},
    {"instruction": "Describe the Himalayan ecosystem.", "input": "", "output": "The Himalayas span five Indian states — Jammu and Kashmir, Ladakh, Himachal Pradesh, Uttarakhand, Sikkim — and extend into Nepal, Bhutan, and Tibet. The ecosystem varies dramatically by altitude from subtropical forests at lower elevations to alpine meadows and permanent snow and ice above 5,000 metres. The region is the source of major rivers including the Ganges, Yamuna, Indus, and Brahmaputra. It is home to snow leopards, red pandas, Himalayan brown bears, and thousands of plant species. The Himalayas act as a climatic barrier protecting India from cold Central Asian winds."},
    {"instruction": "What is the significance of Amritsar?", "input": "", "output": "Amritsar is the spiritual and cultural centre of Sikhism in Punjab. The Golden Temple (Harmandir Sahib) — the holiest Gurdwara in Sikhism — is located here. The temple sits in the middle of a sacred pool called Amrit Sarovar (Pool of Nectar), which gives the city its name. The Golden Temple serves langar (free community meal) to over 100,000 people daily regardless of religion or background. Amritsar is also historically significant as the site of the 1919 Jallianwala Bagh massacre."},
    {"instruction": "What are the major art forms of Kerala?", "input": "", "output": "Kerala has a rich tradition of performing arts. Kathakali is a highly stylised classical dance-drama with elaborate costumes and makeup. Mohiniyattam is a graceful classical dance form performed by women. Theyyam is a ritual folk dance performed at temples in northern Kerala. Koodiyattam is the oldest surviving classical theatre form in India, recognised by UNESCO. Kalaripayattu is an ancient martial art considered the mother of all martial arts. Kerala also has strong traditions in Carnatic and Sopana music."},
    {"instruction": "What is the significance of the Ganga-Yamuna Doab?", "input": "", "output": "The Ganga-Yamuna Doab is the fertile land between the Ganges and Yamuna rivers in Uttar Pradesh and Haryana. Doab means land between two rivers in Persian. It is one of the most agriculturally productive regions in India and one of the most densely populated areas on Earth. Historically it was the heartland of multiple great civilisations and empires including the Maurya and Gupta empires. Important cities in this region include Agra, Mathura, Aligarh, and Meerut."},
    {"instruction": "What is Pongal and how is it celebrated?", "input": "", "output": "Pongal is the most important harvest festival of Tamil Nadu, celebrated in January over four days. The name comes from the Tamil word meaning to boil over, referring to the ritual of boiling newly harvested rice with milk until it overflows — symbolising abundance. Day 1 is Bhogi Pongal for discarding old items. Day 2 is Surya Pongal for worshipping the Sun. Day 3 is Maattu Pongal for honouring cattle. Day 4 is Kaanum Pongal for family gatherings. The festival coincides with Makar Sankranti celebrated across India."},
    {"instruction": "What is the significance of the Eastern Ghats?", "input": "", "output": "The Eastern Ghats are a discontinuous range of hills running parallel to India's eastern coast through Odisha, Andhra Pradesh, and Tamil Nadu. Unlike the continuous Western Ghats, they are fragmented by major rivers including the Godavari, Krishna, and Kaveri. The Eastern Ghats have significant tribal populations including the Kondhs and Savara. The Araku Valley in Andhra Pradesh and the Nilgiri Hills at the southern end where Eastern and Western Ghats meet are notable biodiversity areas. The region is rich in mineral resources."},
    {"instruction": "What is the significance of Mahabalipuram?", "input": "", "output": "Mahabalipuram (Mamallapuram) is a historic town in Tamil Nadu, declared a UNESCO World Heritage Site for its Group of Monuments. Built by the Pallava dynasty in the 7th and 8th centuries CE, the monuments include shore temples built directly on the seafront, five rathas (chariot-shaped temples carved from single rocks), and Arjuna's Penance — one of the largest open-air rock reliefs in the world. It was an important port town in ancient times and shows strong evidence of trade with Southeast Asia and Rome."},
    {"instruction": "Describe the geography of Goa.", "input": "", "output": "Goa is India's smallest state by area, located on the western coast bordering the Arabian Sea. It features a narrow coastal strip of white sandy beaches, the Western Ghats to the east with dense tropical forests, and several rivers including the Mandovi and Zuari. Goa has a distinct Portuguese-influenced culture from 450 years of Portuguese colonial rule ending in 1961. It is India's wealthiest state by per capita GDP and a major tourism destination known for its beaches, churches, spice plantations, and Konkani-Portuguese cuisine."},
    {"instruction": "What is the significance of Nalanda?", "input": "", "output": "Nalanda was one of the greatest centres of learning in the ancient world, located in Bihar. Founded in the 5th century CE, it was a Buddhist monastery and university that attracted students from across Asia including China, Korea, Japan, Tibet, and Southeast Asia. At its peak, Nalanda had over 10,000 students and 2,000 teachers covering subjects from theology and philosophy to medicine, mathematics, and astronomy. It was destroyed by Bakhtiyar Khilji in 1193 CE. A modern Nalanda University was re-established in 2014."},
    {"instruction": "What are the major pilgrimage sites in India?", "input": "", "output": "India has countless pilgrimage sites across religions. For Hindus: the Char Dham (Badrinath, Dwarka, Puri, Rameswaram), Varanasi, Tirupati, Amarnath, and Shirdi. For Muslims: Ajmer Sharif, Fatehpur Sikri, Dargah of Nizamuddin Auliya in Delhi. For Sikhs: Golden Temple Amritsar, Anandpur Sahib, Patna Sahib, Nanded. For Buddhists: Bodh Gaya, Sarnath, Kushinagar, Lumbini (Nepal). For Jains: Palitana in Gujarat, Shravanabelagola in Karnataka, Dilwara temples in Rajasthan."},
    {"instruction": "What is the significance of the Indus Valley Civilisation?", "input": "", "output": "The Indus Valley Civilisation, also called the Harappan Civilisation, was one of the world's three earliest urban civilisations alongside Egypt and Mesopotamia, flourishing from approximately 3300 to 1300 BCE. Major sites include Mohenjo-daro and Harappa in present-day Pakistan, and Lothal, Dholavira, and Kalibangan in India. It featured remarkably advanced urban planning with grid-pattern streets, sophisticated drainage systems, standardised weights, and possibly a writing system yet to be fully deciphered. The civilisation declined around 1900 BCE possibly due to climate change and river shifts."},
    {"instruction": "What is Biryani and what are its regional variations?", "input": "", "output": "Biryani is a fragrant rice dish cooked with spices, and usually meat or vegetables, that has become one of India's most beloved dishes. It originated in the royal kitchens of the Mughal Empire. Major regional variations include Hyderabadi biryani — the most famous, cooked dum style with whole spices. Lucknowi (Awadhi) biryani uses the dum pukht technique with subtle spicing. Kolkata biryani uniquely includes potato. Malabar biryani from Kerala uses short-grain rice with coconut and fried onions. Chettinad biryani from Tamil Nadu is heavily spiced. Each region's biryani reflects its culinary heritage and local ingredients."},
    {"instruction": "What is the significance of Bodh Gaya?", "input": "", "output": "Bodh Gaya is a religious site in Bihar where Siddhartha Gautama, the founder of Buddhism, attained enlightenment under the Bodhi tree around 500 BCE. It is the most important pilgrimage site in Buddhism and a UNESCO World Heritage Site. The Mahabodhi Temple, built near the original Bodhi tree, is the central monument. The site attracts Buddhist pilgrims from across the world especially from Tibet, Thailand, Japan, Sri Lanka, and Myanmar. The Dalai Lama regularly visits Bodh Gaya for teachings."},
    {"instruction": "Describe the geography of the Gangetic Plain.", "input": "", "output": "The Gangetic Plain, or Indo-Gangetic Plain, is one of the world's most fertile and densely populated regions, stretching from Punjab in the west to West Bengal in the east across northern India. It is formed by alluvial deposits from the Himalayan rivers — Ganges, Yamuna, Brahmaputra, and their tributaries. The plain covers states including Punjab, Haryana, Delhi, Uttar Pradesh, Bihar, and West Bengal. It is India's agricultural heartland, producing wheat, rice, and sugarcane. It has been the cradle of Indian civilisation with major ancient cities along its banks."},
    {"instruction": "What is the significance of Konkan coast?", "input": "", "output": "The Konkan coast is the narrow coastal strip along the western coast of India between Mumbai and Goa, covering parts of Maharashtra and Goa. It is bounded by the Arabian Sea to the west and the Western Ghats to the east. The coast receives extremely heavy monsoon rainfall — among the highest in India. It is known for its scenic beaches, seafood cuisine, and Alphonso mangoes. The Konkan Railway, built through extremely challenging terrain, connects Mumbai to Mangalore. The region has a distinct Konkani language and culture."},
    {"instruction": "What are the major festivals of North India?", "input": "", "output": "North India celebrates numerous major festivals. Diwali and Holi are celebrated with great enthusiasm. Dussehra features massive Ravan effigies burned in cities like Delhi and Varanasi. Janmashtami marks Krishna's birth with special celebrations in Mathura and Vrindavan. Chhath Puja is a major festival in Bihar and eastern UP where devotees worship the sun god. The Kumbh Mela at Prayagraj and Haridwar draws tens of millions. Lohri marks the winter solstice in Punjab. Navratri features nine nights of Garba and Dandiya in Gujarat and Durga Puja worship in Bengal."},
]

print(f"Dataset now has {len(dataset)} examples")

Dataset now has 51 examples


In [6]:
from datasets import Dataset

def format_instruction(example):
    if example["input"]:
        text = f"""### Instruction:
{example["instruction"]}

### Input:
{example["input"]}

### Response:
{example["output"]}"""
    else:
        text = f"""### Instruction:
{example["instruction"]}

### Response:
{example["output"]}"""
    return {"text": text}

formatted = [format_instruction(ex) for ex in dataset]
hf_dataset = Dataset.from_list(formatted)
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 45
    })
    test: Dataset({
        features: ['text'],
        num_rows: 6
    })
})


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 3,212,749,824


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,587,520 || all params: 3,217,337,344 || trainable%: 0.1426


In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
    ),
    train_dataset=hf_dataset["train"],
    eval_dataset=hf_dataset["test"],
    processing_class=tokenizer,
)

print("Trainer ready. Starting training...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Trainer ready. Starting training...


Epoch,Training Loss,Validation Loss
1,No log,1.884485
2,1.727866,1.677690
3,1.727866,1.561823


TrainOutput(global_step=18, training_loss=1.5933751000298395, metrics={'train_runtime': 52.8017, 'train_samples_per_second': 2.557, 'train_steps_per_second': 0.341, 'total_flos': 301007028307968.0, 'train_loss': 1.5933751000298395})

In [11]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

def ask(question):
    prompt = f"""### Instruction:
{question}

### Response:
"""
    output = pipe(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        temperature=1.0,
        repetition_penalty=1.1
    )
    response = output[0]["generated_text"]
    response = response.split("### Response:")[-1].strip()
    print(f"Q: {question}")
    print(f"A: {response}")
    print()

ask("What is the significance of the Ganges river?")
ask("What is the national animal of India?")
ask("What is yoga and where did it originate?")

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the significance of the Ganges river?
A: The Ganges, also known as the Ganga, is a sacred river in India and one of its most important rivers. It originates from the Gangotri Glacier in the Himalayas and flows through several states before emptying into the Bay of Bengal. The Ganges has significant cultural, spiritual, and economic importance for millions of people living along its banks. Many Hindus consider it a sacred pilgrimage site, while others rely on it for drinking water, irrigation, and fishing.



Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the national animal of India?
A: The national animal of India is the Bengal tiger.

Q: What is yoga and where did it originate?
A: Yoga originated in ancient India over 5,000 years ago. It involves physical postures (asanas), breathing techniques (pranayama), and meditation to promote physical, mental, and spiritual well-being.



In [12]:
ask("What is the significance of the Red Fort?")
ask("What are the major crops grown in Punjab?")
ask("Who was the first Prime Minister of India?")

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the significance of the Red Fort?
A: The Red Fort, also known as Lal Qila, was built by the Mughal Emperor Shah Jahan in 1648. It served as the imperial palace for nearly a century and was used for various administrative purposes. The fort's name comes from its red sandstone walls. Today, it is one of India's most popular tourist attractions and a UNESCO World Heritage Site.



Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the major crops grown in Punjab?
A: Punjab is known for its rice, wheat and sugarcane.

Q: Who was the first Prime Minister of India?
A: Jawaharlal Nehru.



In [13]:
model.save_pretrained("./india-llm")
tokenizer.save_pretrained("./india-llm")
print("Model saved to ./india-llm")

Model saved to ./india-llm


In [14]:
import shutil
shutil.make_archive("india-llm", "zip", "./india-llm")
print("Zipped successfully")

Zipped successfully


In [15]:
from google.colab import files
files.download("india-llm.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>